In [10]:
import sys
import os

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "..", "..", ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.data.preprocessing import data_pipeline
from utils.nn.QuantumNN import QuantumNN

print(f"Successfully imported from {project_root}")

Successfully imported from /Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design


In [11]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import openml
from pathlib import Path
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

import qiskit
from qiskit import QuantumCircuit
from qiskit.circuit.library import TwoLocal, ZZFeatureMap
from qiskit.primitives import StatevectorSampler, Sampler
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.gradients import (
    ParamShiftSamplerGradient,
    SPSASamplerGradient,
)
from qiskit.transpiler import PassManager

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)


In [12]:
# TODO: RERUN SEED 0 FOR [6,8] QUBITS

In [13]:
# CONFIGS

USE_GPU = False
VERBOSE_ON_FIT = False

SEEDS = [4] # [1, 2, 3, 4]  # For reproducibility

N_QUBITS = [6,8] # [2,4,6,8]

OPENML_DATASET_IDS = {
    "iris": (61, 4),
    "wine": (187, 13),
    "diabetes": (37, 8),
}
DATASET_NAMES = list(OPENML_DATASET_IDS.keys())

# Common training parameters
DO_PCA = True

BATCH_SIZE = 32
EPOCHS_CLASSICAL = 100  # Reduced for quick testing; use your value e.g., 100
EPOCHS_QUANTUM = 30  # Reduced for quick testing; use your value e.g., 50
CLASSICAL_LR = 0.01
QUANTUM_LR = 0.05

In [14]:
def set_seed(seed):
    """Set the random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [15]:
from utils.ansatze.AnsatzExtractor import extract_and_store_model_schema

In [ ]:
from utils.ansatze.HierarchicalGatewise import HierarchicalGatewiseAnsatz
from utils.ansatze.HierarchicalLayerwise import HierarchicalLayerwiseAnsatz

from expressibility_proxy import calculate_expressibility_proxy
from path_proxy import calculate_path_proxy

def run_tf_qas(ansatze_class, ansatz_args, S, R, K):
    """
    Orchestrates the full two-stage Training-Free QAS algorithm.

    Args:
        ansatze_class: The class to use for generating circuits (e.g., HierarchicalGatewiseAnsatz).
        ansatz_args: A dictionary of arguments for the ansatz class.
        S: The initial number of circuits to sample.
        R: The number of circuits to keep after the path-proxy filter.
        K: The final number of top circuits to return.

    Returns:
        A list of the top K circuits, ranked by expressibility.
    """
    print(f"--- Starting TF-QAS ---")
    print(f"Initial circuits (S): {S}, Path-proxy filtered (R): {R}, Final circuits (K): {K}")

    # --- Step 1: Randomly sample S circuits from the search space ---
    # Each circuit is stored as a dictionary to keep its data together.
    initial_circuits = []
    for i in range(S):
        ansatz = ansatze_class(**ansatz_args, seed=i)
        initial_circuits.append({"seed": i, "ansatz": ansatz})
    print(f"\nStep 1: Generated {len(initial_circuits)} initial circuits.")

    # --- Step 2 & 3: Filter down to R circuits using the path-based proxy ---
    for circuit_data in initial_circuits:
        path_count = calculate_path_proxy(circuit_data["ansatz"].get_ansatz())
        circuit_data["path_count"] = path_count
    
    # Sort by path_count (descending, as larger is better)
    sorted_by_path = sorted(initial_circuits, key=lambda x: x["path_count"], reverse=True)
    
    top_R_circuits = sorted_by_path[:R]
    print(f"Step 2: Filtered down to the top {len(top_R_circuits)} circuits based on path count.")
    print(f"Best path counts: {[c['path_count'] for c in top_R_circuits[:5]]}...")

    # --- Step 4 & 5: Rank the R circuits by expressibility and return the top K ---
    for circuit_data in top_R_circuits:
        expressibility = calculate_expressibility_proxy(circuit_data["ansatz"].get_ansatz(), n_samples=200) # Using 200 for speed
        circuit_data["expressibility"] = expressibility

    # Sort by expressibility (descending, as a score closer to 0 is better)
    sorted_by_expressibility = sorted(top_R_circuits, key=lambda x: x["expressibility"], reverse=True)

    top_K_circuits = sorted_by_expressibility[:K]
    print(f"\nStep 3: Ranked by expressibility and selected the final {len(top_K_circuits)} circuits.")
    print(f"Best expressibility scores: {[c['expressibility'] for c in top_K_circuits]}")

    return top_K_circuits




In [17]:
# number of qubits
# number of gates
# log best everything

In [18]:
for seed in SEEDS:
    set_seed(seed)
    print(f"\n\n=== Running for Seed: {seed} ===")
    for n_qubits in N_QUBITS:
        print(f"\n\n=== Running for {n_qubits} Qubits ===")
        for dataset in DATASET_NAMES:

            openml_dataset_id = OPENML_DATASET_IDS[dataset][0]
            n_features = OPENML_DATASET_IDS[dataset][1]

            if n_qubits > n_features:
                print(f"[SKIP] {dataset} has only {n_features} features, cannot run with {n_qubits} qubits.")
                continue

            if dataset == "wine" and n_qubits == 6:
                print(f"[SKIP] Skipping wine dataset for 6 qubits, already done.")
                continue

            # 1. Load and preprocess data
            print(f"\n[INFO] Loading data: {dataset}, number of features: {n_qubits}")
            _, quantum_data, input_dim, output_dim, is_multiclass = (
                data_pipeline(
                    openml_dataset_id,
                    batch_size=BATCH_SIZE,
                    do_pca=DO_PCA,
                    use_gpu=USE_GPU,
                    n_components=n_qubits,
                    seed=seed,
                )
            )
            (
                x_train_q,
                x_val_q,
                x_test_q,
                y_train_q,
                y_val_q,
                y_test_q,
                train_loader_q,
                val_loader_q,
                test_loader_q,
            ) = quantum_data

            S = 50000  # Initial sample size (Paper uses 50000)
            R = 5000   # Number to keep after path filter (Paper uses 5000)
            K = 50    # Final number of top circuits to return (Paper uses ~100)

            mode = 'gate'  # Choose 'gate' or 'layer'
            ansatz_args = {"n_qubits": n_qubits}
            if mode == 'gate':
                ansatz_args['n_gates'] = 24
                temporal_bias=0.75
                spatial_bias=1.0
                ansatz_args['temporal_bias'] = temporal_bias
                ansatz_args['spatial_bias'] = spatial_bias
                data_obj = run_tf_qas(HierarchicalGatewiseAnsatz, ansatz_args, S, R, K)
            elif mode == 'layer':
                ansatz_args['depth'] = 4
                data_obj = run_tf_qas(HierarchicalLayerwiseAnsatz, ansatz_args, S, R, K)
            else:
                raise ValueError("Mode must be either 'gate' or 'layer'")

            # write circuit to file
            base_dir = f"results/{dataset}/seed_{seed}"
            os.makedirs(base_dir, exist_ok=True)

            circuit_fp = f"{base_dir}/{n_qubits}q_circuit.qpy"
            params_fp = f"{base_dir}/{n_qubits}q_params.json"
            res_fp = f"{base_dir}/{n_qubits}q_results.txt"

            best_circuit = None
            best_test_f1 = 0

            for i, ansatz_data in enumerate(data_obj):   
                start_time = time.time() 
                print(f"\n=== Training with Circuit {i+1}/{len(data_obj)} ===")
                circuit = ansatz_data["ansatz"]
                set_seed(seed)
                model_q = QuantumNN(
                    ansatz=circuit.get_ansatz(),
                    n_qubits=input_dim,
                    num_classes=output_dim,
                    use_gpu=USE_GPU, 
                    gradient_method="guided_spsa"
                )

                # Optimizer and Scheduler for Quantum Model
                optimizer_q = optim.Adam(model_q.parameters(), lr=QUANTUM_LR)
                scheduler_q = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer_q,
                    mode="min",
                    factor=0.3,
                    patience=max(1, EPOCHS_QUANTUM // 5),
                    min_lr=1e-4,
                    verbose=False,
                )

                start_time_q = time.time()
                model_q.fit(
                    train_loader_q,
                    val_loader_q,
                    epochs=EPOCHS_QUANTUM,
                    optimizer=optimizer_q,
                    scheduler=scheduler_q,
                    verbose=VERBOSE_ON_FIT,
                    eval_every=2
                )  # Set verbose
                end_time_q = time.time()
                avg_time_epoch_q = (
                    (end_time_q - start_time_q) / EPOCHS_QUANTUM
                    if EPOCHS_QUANTUM > 0
                    else 0
                )

                eval_output_q = model_q.evaluate(test_loader_q, verbose=False)
                test_metrics_q_dict = {
                    "test_acc": eval_output_q[1],
                    "test_loss": eval_output_q[0],
                    "test_prec": eval_output_q[2],
                    "test_rec": eval_output_q[3],
                    "test_f1": eval_output_q[4]
                }
                print(
                    f"Test Results: Loss={test_metrics_q_dict['test_loss']:.4f}, Acc={test_metrics_q_dict['test_acc']:.4f}, F1={test_metrics_q_dict['test_f1']:.4f}"
                )

                if test_metrics_q_dict['test_f1'] > best_test_f1:
                    best_test_f1 = test_metrics_q_dict['test_f1']
                    associated_metrics = test_metrics_q_dict
                    best_circuit = circuit
                    print(f"[INFO] New best circuit found with F1: {best_test_f1:.4f}")
                    print(f"[INFO] Associated Metrics: {associated_metrics}")
                    # best_circuit.draw()

                    extract_and_store_model_schema(
                        best_circuit,
                        model_q,  # Pass the (possibly trained) model_q if weights are desired
                        circuit_fp,
                        params_fp,
                    )

                    with open(res_fp, 'w') as f:
                        f.write(f"Acc: {associated_metrics['test_acc']:.4f}\n")
                        f.write(f"Prec: {associated_metrics['test_prec']:.4f}\n")
                        f.write(f"Rec: {associated_metrics['test_rec']:.4f}\n")
                        f.write(f"F1 Score: {best_test_f1:.4f}\n")
                        
                print(f"[INFO] Time per epoch: {avg_time_epoch_q:.2f} seconds")

            print(f"\n[INFO] Best Circuit for {dataset}: {best_circuit}")
            print(f"[INFO] Best Test F1 Score for {dataset}: {best_test_f1:.4f}")



=== Running for Seed: 4 ===


=== Running for 6 Qubits ===
[SKIP] iris has only 4 features, cannot run with 6 qubits.
[SKIP] Skipping wine dataset for 6 qubits, already done.

[INFO] Loading data: diabetes, number of features: 6
Dataset name: diabetes
Total features: 8, Dropped Categorical features: 0, Remaining features: 8
Classical data reduced to 6 dimensions using PCA.
Configuring DataLoader for CPU.
Quantum data reduced to 6 dimensions using PCA.
Configuring DataLoader for CPU.
--- Starting TF-QAS ---
Initial circuits (S): 50000, Path-proxy filtered (R): 5000, Final circuits (K): 50

Step 1: Generated 50000 initial circuits.
Step 2: Filtered down to the top 5000 circuits based on path count.
Best path counts: [176, 139, 139, 136, 134]...

Step 3: Ranked by expressibility and selected the final 50 circuits.
Best expressibility scores: [-0.053827532742251126, -0.06222198366305243, -0.07370057248783175, -0.08171048217301748, -0.08961282059099906, -0.09279097092015752, -0.0935366772

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:59: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Test Results: Loss=0.6376, Acc=0.6169, F1=0.3218
[INFO] New best circuit found with F1: 0.3218
[INFO] Associated Metrics: {'test_acc': 0.6168831168831169, 'test_loss': 0.6375689854869595, 'test_prec': 0.4117647058823529, 'test_rec': 0.2641509433962264, 'test_f1': 0.32183908045977005}
Model schema saved to results/diabetes/seed_4/6q_circuit.qpy
Model parameters saved to results/diabetes/seed_4/6q_params.json
[INFO] Time per epoch: 41.72 seconds

=== Training with Circuit 2/50 ===
Initialized QuantumNN on device: cpu


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:59: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Test Results: Loss=0.6853, Acc=0.5584, F1=0.3585
[INFO] New best circuit found with F1: 0.3585
[INFO] Associated Metrics: {'test_acc': 0.5584415584415584, 'test_loss': 0.6852604638446461, 'test_prec': 0.3584905660377358, 'test_rec': 0.3584905660377358, 'test_f1': 0.3584905660377358}
Model schema saved to results/diabetes/seed_4/6q_circuit.qpy
Model parameters saved to results/diabetes/seed_4/6q_params.json
[INFO] Time per epoch: 41.82 seconds

=== Training with Circuit 3/50 ===
Initialized QuantumNN on device: cpu


...

Test Results: Loss=0.6847, Acc=0.5519, F1=0.4651
[INFO] Time per epoch: 48.21 seconds

=== Training with Circuit 49/50 ===
Initialized QuantumNN on device: cpu


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:59: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Test Results: Loss=0.6927, Acc=0.4935, F1=0.4000
[INFO] Time per epoch: 59.30 seconds

=== Training with Circuit 50/50 ===
Initialized QuantumNN on device: cpu


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:59: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Test Results: Loss=0.6707, Acc=0.5779, F1=0.3011
[INFO] Time per epoch: 58.42 seconds

[INFO] Best Circuit for diabetes: <utils.ansatze.HierarchicalGatewise.HierarchicalGatewiseAnsatz object at 0x36ebfc7d0>
[INFO] Best Test F1 Score for diabetes: 0.5736
